In [5]:
pip install holidays


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\natas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 2.8 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.1 MB/s eta 0:00:00
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.2
    Uninstalling python-dateutil-2.8.2:
      Successfully uninstalled python-dateutil-2.8.2


In [ ]:

# =========================
# IMPORTS & DEPENDENCIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU
from tensorflow.keras.optimizers import Adam, Nadam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import holidays


C:\Users\natas\AppData\Local\Temp\ipykernel_30344\523810513.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


ModuleNotFoundError: No module named 'holidays'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import holidays

# =========================
# 1. LOAD
# =========================
df = pd.read_excel("/content/drive/MyDrive/Diploma/RawData/all_clean+more_weather.xlsx", header=None)

df.columns = [
    'date', 'hour_range', 'forecast', 'actual',
    'temperature_2m', 'relative_humidity_2m', 'weather_code',
    'cloud_cover', 'wind_speed_10m', 'dew_point_2m',
    'apparent_temperature', 'surface_pressure', 'wind_speed_100m',
    'wind_direction_10m', 'soil_temperature_0_to_7cm',
    'soil_moisture_0_to_7cm'
]

df['is_date'] = df['date'].astype(str).str.match(r'\d{4}-\d{2}-\d{2}')
df['date'] = df['date'].where(df['is_date']).ffill()

df = df[pd.to_numeric(df['actual'], errors='coerce').notnull()].copy()

df['hour'] = df['hour_range'].str.split('--').str[0].astype(int)
df['datetime'] = pd.to_datetime(df['date']) + pd.to_timedelta(df['hour'], unit='h')

df = df.sort_values('datetime').reset_index(drop=True)

# =========================
# 2. TIME FEATURES
# =========================
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

df['dow'] = df['datetime'].dt.dayofweek
df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)

df['month'] = df['datetime'].dt.month
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

# =========================
# 3. HOLIDAYS
# =========================
ru_holidays = holidays.Russia(years=df['datetime'].dt.year.unique())

df['is_weekend'] = df['dow'].isin([5,6]).astype(int)
df['is_holiday'] = df['datetime'].dt.date.apply(lambda x: int(x in ru_holidays))
df['is_special_day'] = ((df['is_weekend'] + df['is_holiday']) > 0).astype(int)

# =========================
# 4. LAGS + ROLLING (only past info)
# =========================
df['lag_1'] = df['actual'].shift(1)
df['lag_24'] = df['actual'].shift(24)
df['lag_168'] = df['actual'].shift(168)

df['roll_24'] = df['actual'].shift(1).rolling(24).mean()
df['roll_168'] = df['actual'].shift(1).rolling(168).mean()

# =========================
# 5. FEATURES
# =========================
weather_features = [
    'temperature_2m','relative_humidity_2m','cloud_cover',
    'wind_speed_10m','surface_pressure','apparent_temperature',
    'dew_point_2m','wind_speed_100m','soil_temperature_0_to_7cm',
    'weather_code','wind_direction_10m','soil_moisture_0_to_7cm'
]

features = [
    'hour_sin','hour_cos',
    'dow_sin','dow_cos',
    'month_sin','month_cos',
    'is_special_day',
    'lag_1','lag_24','lag_168',
    'roll_24','roll_168'
] + weather_features

target = 'actual'

# =========================
# 6. CLEAN
# =========================
df = df.dropna().reset_index(drop=True)

# =========================
# 7. TIME SPLIT (NO LEAKAGE)
# =========================
n = len(df)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df[:train_end].copy()
val_df = df[train_end:val_end].copy()
test_df = df[val_end:].copy()

# =========================
# 8. SCALING (FIT ONLY ON TRAIN)
# =========================
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_raw = scaler_X.fit_transform(train_df[features])
X_val_raw = scaler_X.transform(val_df[features])
X_test_raw = scaler_X.transform(test_df[features])

y_train_raw = scaler_y.fit_transform(train_df[[target]])
y_val_raw = scaler_y.transform(val_df[[target]])
y_test_raw = scaler_y.transform(test_df[[target]])

# =========================
# 9. SEQUENCE BUILDER
# =========================
LOOKBACK = 168   # 7 days
HORIZON = 24      # next day

def create_sequences(X, y, lookback=168, horizon=24):
    Xs, ys = [], []
    for i in range(len(X) - lookback - horizon):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback:i+lookback+horizon])
    return np.array(Xs), np.array(ys)

# =========================
# 10. BUILD SETS
# =========================
X_train, y_train = create_sequences(X_train_raw, y_train_raw, LOOKBACK, HORIZON)
X_val, y_val = create_sequences(X_val_raw, y_val_raw, LOOKBACK, HORIZON)
X_test, y_test = create_sequences(X_test_raw, y_test_raw, LOOKBACK, HORIZON)

# =========================
# 11. SHAPES CHECK
# =========================
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

n_features = X_train.shape[2]

LOOKBACK = X_train.shape[1]
HORIZON = y_train.shape[1]

# =========================
# MODEL: ENCODER-DECODER LSTM
# =========================
model = models.Sequential([
    layers.Input(shape=(LOOKBACK, n_features)),

    # -------- Encoder --------
    layers.LSTM(128, return_sequences=False),
    layers.Dropout(0.2),

    # -------- Bridge --------
    layers.RepeatVector(HORIZON),

    # -------- Decoder --------
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(64, return_sequences=True),

    layers.TimeDistributed(layers.Dense(32, activation='relu')),
    layers.TimeDistributed(layers.Dense(1))
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse',
    metrics=['mae']
)

model.summary()

# =========================
# CALLBACKS
# =========================
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-5
)

# =========================
# TRAIN
# =========================
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# =========================
# PREDICT
# =========================
y_pred = model.predict(X_test)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# 1. INVERSE SCALE
# =========================
y_test_inv = scaler_y.inverse_transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

y_pred_inv = scaler_y.inverse_transform(
    y_pred.reshape(-1, 1)
).reshape(y_pred.shape)

# =========================
# 2. FLATTENED METRICS (global)
# =========================
y_true_flat = y_test_inv.flatten()
y_pred_flat = y_pred_inv.flatten()

mae = mean_absolute_error(y_true_flat, y_pred_flat)
rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))

mape = np.mean(np.abs((y_true_flat - y_pred_flat) / (y_true_flat + 1e-8))) * 100

smape = np.mean(
    2.0 * np.abs(y_pred_flat - y_true_flat) /
    (np.abs(y_true_flat) + np.abs(y_pred_flat) + 1e-8)
) * 100

print("\n===== GLOBAL METRICS =====")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAPE  : {mape:.2f}%")
print(f"sMAPE : {smape:.2f}%")

# =========================
# 3. PER-HORIZON METRICS (T+1 ... T+24)
# =========================
horizon = y_test.shape[1]

mae_per_t = []
rmse_per_t = []
mape_per_t = []

for t in range(horizon):
    yt = y_test_inv[:, t, 0]
    yp = y_pred_inv[:, t, 0]

    mae_t = mean_absolute_error(yt, yp)
    rmse_t = np.sqrt(mean_squared_error(yt, yp))
    mape_t = np.mean(np.abs((yt - yp) / (yt + 1e-8))) * 100

    mae_per_t.append(mae_t)
    rmse_per_t.append(rmse_t)
    mape_per_t.append(mape_t)

print("\n===== PER-HORIZON METRICS =====")
for t in range(horizon):
    print(
        f"T+{t+1:02d}h | "
        f"MAE: {mae_per_t[t]:.3f} | "
        f"RMSE: {rmse_per_t[t]:.3f} | "
        f"MAPE: {mape_per_t[t]:.2f}%"
    )

# =========================
# 4. SUMMARY INSIGHT (useful for analysis)
# =========================
print("\n===== SUMMARY =====")
print(f"Best step (MAE): T+{np.argmin(mae_per_t)+1}")
print(f"Worst step (MAE): T+{np.argmax(mae_per_t)+1}")

In [ ]:
# =========================
# 6. BASELINE: SEASONAL NAIVE
# =========================
# Baseline: use load from same hour of previous day (lag_24)

y_baseline = np.zeros_like(y_pred_inv)

for i in range(len(y_test_inv)):
    for t in range(HORIZON):
        # Get the actual value 24 hours back from the first hour of the test sequence
        # X_test contains 168 hours of history
        # We want the value 24 hours before the first prediction hour
        baseline_idx = LOOKBACK - 24 + t
        if baseline_idx >= 0:
            # Get from the test data directly
            y_baseline[i, t, 0] = y_test_inv[i, t, 0] if t < len(y_test_inv[i]) else 0

# Alternative simpler baseline: use last known value from lookback window shifted by 24
# For each test sequence, take the value from 24 hours before start of prediction
def get_seasonal_naive_baseline(y_test_data, lookback=168, horizon=24):
    """
    For each prediction sequence, baseline is the value from 24 hours earlier
    y_test_data shape: (n_sequences, horizon, 1)
    """
    baseline = np.zeros_like(y_test_data)
    
    # Since we need to reference the lookback window, we'll use the actual values
    # from test_df at appropriate lags
    for seq_idx in range(len(y_test_data)):
        for t in range(horizon):
            actual_test_idx = val_end + (lookback - HORIZON) + seq_idx * HORIZON + t
            lag_24_idx = actual_test_idx - 24
            
            if lag_24_idx >= 0 and lag_24_idx < len(df):
                baseline[seq_idx, t, 0] = df.iloc[lag_24_idx][target]
    
    return baseline

y_baseline = get_seasonal_naive_baseline(y_test_inv, LOOKBACK, HORIZON)

# =========================
# 7. BASELINE METRICS
# =========================
y_baseline_flat = y_baseline.flatten()

mae_baseline = mean_absolute_error(y_true_flat, y_baseline_flat)
rmse_baseline = np.sqrt(mean_squared_error(y_true_flat, y_baseline_flat))
mape_baseline = np.mean(np.abs((y_true_flat - y_baseline_flat) / (y_true_flat + 1e-8))) * 100
smape_baseline = np.mean(
    2.0 * np.abs(y_baseline_flat - y_true_flat) /
    (np.abs(y_true_flat) + np.abs(y_baseline_flat) + 1e-8)
) * 100

print("\n===== BASELINE (SEASONAL NAIVE - LAG 24) =====")
print(f"MAE   : {mae_baseline:.4f}")
print(f"RMSE  : {rmse_baseline:.4f}")
print(f"MAPE  : {mape_baseline:.2f}%")
print(f"sMAPE : {smape_baseline:.2f}%")

# =========================
# 8. MODEL vs BASELINE COMPARISON
# =========================
improvement_mae = ((mae_baseline - mae) / mae_baseline) * 100
improvement_rmse = ((rmse_baseline - rmse) / rmse_baseline) * 100
improvement_mape = ((mape_baseline - mape) / mape_baseline) * 100

print("\n===== MODEL IMPROVEMENT vs BASELINE =====")
print(f"MAE improvement   : {improvement_mae:.2f}%")
print(f"RMSE improvement  : {improvement_rmse:.2f}%")
print(f"MAPE improvement  : {improvement_mape:.2f}%")

if improvement_mae > 0:
    print("✓ Model outperforms baseline!")
else:
    print("✗ Baseline performs better - consider tuning model")


In [ ]:
# =========================
# 9. VISUALIZATION: ACTUAL vs PREDICTED vs BASELINE
# =========================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Select a few representative test sequences to visualize
n_samples_to_plot = 5

fig, axes = plt.subplots(n_samples_to_plot, 1, figsize=(14, 4*n_samples_to_plot))
if n_samples_to_plot == 1:
    axes = [axes]

for plot_idx in range(n_samples_to_plot):
    ax = axes[plot_idx]
    
    hours = np.arange(HORIZON)
    
    actual = y_test_inv[plot_idx, :, 0]
    predicted = y_pred_inv[plot_idx, :, 0]
    baseline = y_baseline[plot_idx, :, 0]
    
    ax.plot(hours, actual, 'o-', linewidth=2, label='Actual', color='green', markersize=6)
    ax.plot(hours, predicted, 's--', linewidth=2, label='LSTM Prediction', color='blue', markersize=5)
    ax.plot(hours, baseline, '^--', linewidth=1.5, label='Baseline (Lag-24)', 
            color='orange', markersize=5, alpha=0.7)
    
    ax.set_xlabel('Hour Ahead (T+h)', fontsize=11)
    ax.set_ylabel('Electricity Load', fontsize=11)
    ax.set_title(f'Test Sequence {plot_idx+1}: Actual vs Predicted vs Baseline', fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(hours[::2])

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization: forecast_comparison.png")


In [ ]:
# =========================
# 10. ERROR ANALYSIS: PER-HORIZON COMPARISON
# =========================
mae_per_t_baseline = []
rmse_per_t_baseline = []

for t in range(HORIZON):
    yt = y_test_inv[:, t, 0]
    yb = y_baseline[:, t, 0]
    
    mae_t = mean_absolute_error(yt, yb)
    rmse_t = np.sqrt(mean_squared_error(yt, yb))
    
    mae_per_t_baseline.append(mae_t)
    rmse_per_t_baseline.append(rmse_t)

# Plot per-horizon error comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

hours = np.arange(1, HORIZON+1)

# MAE comparison
ax1.plot(hours, mae_per_t, 'o-', linewidth=2, label='LSTM Model', color='blue', markersize=7)
ax1.plot(hours, mae_per_t_baseline, 's--', linewidth=2, label='Baseline', 
         color='orange', markersize=6, alpha=0.7)
ax1.fill_between(hours, mae_per_t, mae_per_t_baseline, alpha=0.2, color='green')
ax1.set_xlabel('Hours Ahead', fontsize=11)
ax1.set_ylabel('MAE', fontsize=11)
ax1.set_title('MAE Per Forecast Horizon', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(hours[::2])

# RMSE comparison
ax2.plot(hours, rmse_per_t, 'o-', linewidth=2, label='LSTM Model', color='blue', markersize=7)
ax2.plot(hours, rmse_per_t_baseline, 's--', linewidth=2, label='Baseline', 
         color='orange', markersize=6, alpha=0.7)
ax2.fill_between(hours, rmse_per_t, rmse_per_t_baseline, alpha=0.2, color='green')
ax2.set_xlabel('Hours Ahead', fontsize=11)
ax2.set_ylabel('RMSE', fontsize=11)
ax2.set_title('RMSE Per Forecast Horizon', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(hours[::2])

plt.tight_layout()
plt.savefig('error_per_horizon_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization: error_per_horizon_comparison.png")


In [ ]:
# =========================
# 11. RESULTS SUMMARY TABLE
# =========================
import pandas as pd

results_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)', 'sMAPE (%)'],
    'LSTM Model': [f"{mae:.4f}", f"{rmse:.4f}", f"{mape:.2f}", f"{smape:.2f}"],
    'Baseline': [f"{mae_baseline:.4f}", f"{rmse_baseline:.4f}", f"{mape_baseline:.2f}", f"{smape_baseline:.2f}"],
    'Improvement (%)': [f"{improvement_mae:.2f}", f"{improvement_rmse:.2f}", f"{improvement_mape:.2f}", "-"]
})

print("\n" + "="*80)
print("FINAL RESULTS COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Save to CSV
results_df.to_csv('model_results_comparison.csv', index=False)
print(f"✓ Results saved to: model_results_comparison.csv")


In [ ]:
# =========================
# 12. TRAINING HISTORY VISUALIZATION
# =========================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], 'o-', linewidth=2, label='Train Loss', color='blue', markersize=5)
ax1.plot(history.history['val_loss'], 's--', linewidth=2, label='Validation Loss', 
         color='orange', markersize=5)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss (MSE)', fontsize=11)
ax1.set_title('Training History: Loss', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# MAE
ax2.plot(history.history['mae'], 'o-', linewidth=2, label='Train MAE', color='blue', markersize=5)
ax2.plot(history.history['val_mae'], 's--', linewidth=2, label='Validation MAE', 
         color='orange', markersize=5)
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('MAE', fontsize=11)
ax2.set_title('Training History: MAE', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization: training_history.png")


In [ ]:
# =========================
# 13. EXPORT PREDICTIONS
# =========================

# Create predictions DataFrame with timestamps
test_start_idx = val_end + LOOKBACK
predictions_list = []

for seq_idx in range(len(y_test_inv)):
    base_idx = test_start_idx + seq_idx * HORIZON
    
    for t in range(HORIZON):
        idx = base_idx + t
        if idx < len(df):
            predictions_list.append({
                'datetime': df.iloc[idx]['datetime'],
                'actual': y_test_inv[seq_idx, t, 0],
                'predicted': y_pred_inv[seq_idx, t, 0],
                'baseline': y_baseline[seq_idx, t, 0],
                'error_model': abs(y_test_inv[seq_idx, t, 0] - y_pred_inv[seq_idx, t, 0]),
                'error_baseline': abs(y_test_inv[seq_idx, t, 0] - y_baseline[seq_idx, t, 0])
            })

predictions_df = pd.DataFrame(predictions_list)
predictions_df.to_csv('predictions_detailed.csv', index=False)
print(f"✓ Exported detailed predictions to: predictions_detailed.csv")
print(f"  Total predictions: {len(predictions_df)}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10).to_string(index=False))


In [ ]:
# =========================
# 14. MODEL OPTIMIZATION RECOMMENDATIONS
# =========================

print("\n" + "="*80)
print("MODEL OPTIMIZATION RECOMMENDATIONS")
print("="*80)

recommendations = []

# Check if model outperforms baseline
if improvement_mae > 0:
    recommendations.append(f"✓ Model outperforms baseline by {improvement_mae:.2f}% (MAE)")
else:
    recommendations.append(f"⚠ Baseline performs better. Consider:")
    recommendations.append("  • Increase model complexity (more LSTM units/layers)")
    recommendations.append("  • Increase training epochs")
    recommendations.append("  • Add more relevant features")
    recommendations.append("  • Adjust learning rate")

# Check error growth over horizon
error_growth = mape_per_t[-1] - mape_per_t[0]
if error_growth > 5:
    recommendations.append(f"\n⚠ High error growth over horizon ({error_growth:.2f}%)")
    recommendations.append("  • Consider attention mechanism to weight recent timesteps")
    recommendations.append("  • Try Transformer-based architecture")

# Suggest hyperparameter tuning
if rmse > 10:  # Assuming this is a high error threshold
    recommendations.append("\n• Try hyperparameter tuning:")
    recommendations.append("  • Experiment with LSTM units: [64, 128, 256]")
    recommendations.append("  • Adjust dropout rate: [0.1, 0.3, 0.5]")
    recommendations.append("  • Try different batch sizes: [16, 32, 64]")
    recommendations.append("  • Learning rate scheduling with ReduceLROnPlateau")

recommendations.append("\n• Additional improvements:")
recommendations.append("  • Use Temporal Fusion Transformer (TFT) for better multi-horizon forecasting")
recommendations.append("  • Implement ensembling with multiple models")
recommendations.append("  • Add adversarial training or uncertainty quantification")
recommendations.append("  • Incorporate external data (holidays, special events)")

for rec in recommendations:
    print(rec)

print("="*80)


In [ ]:
# =========================
# 15. ALTERNATIVE MODEL: GRU ENCODER-DECODER
# =========================
from tensorflow.keras.layers import GRU

print("\n" + "="*80)
print("TRAINING ALTERNATIVE MODEL: GRU ENCODER-DECODER")
print("="*80 + "\n")

# Build GRU model
model_gru = models.Sequential([
    layers.Input(shape=(LOOKBACK, n_features)),
    
    # Encoder
    layers.GRU(128, return_sequences=False),
    layers.Dropout(0.2),
    
    # Bridge
    layers.RepeatVector(HORIZON),
    
    # Decoder
    layers.GRU(128, return_sequences=True),
    layers.Dropout(0.2),
    
    layers.GRU(64, return_sequences=True),
    
    layers.TimeDistributed(layers.Dense(32, activation='relu')),
    layers.TimeDistributed(layers.Dense(1))
])

model_gru.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse',
    metrics=['mae']
)

# Callbacks
early_stop_gru = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr_gru = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-5
)

# Train
history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_gru, reduce_lr_gru],
    verbose=1
)

# Predict
y_pred_gru = model_gru.predict(X_test)

# Inverse transform
y_pred_gru_inv = scaler_y.inverse_transform(
    y_pred_gru.reshape(-1, 1)
).reshape(y_pred_gru.shape)

# Metrics
y_pred_gru_flat = y_pred_gru_inv.flatten()

mae_gru = mean_absolute_error(y_true_flat, y_pred_gru_flat)
rmse_gru = np.sqrt(mean_squared_error(y_true_flat, y_pred_gru_flat))
mape_gru = np.mean(np.abs((y_true_flat - y_pred_gru_flat) / (y_true_flat + 1e-8))) * 100
smape_gru = np.mean(
    2.0 * np.abs(y_pred_gru_flat - y_true_flat) /
    (np.abs(y_true_flat) + np.abs(y_pred_gru_flat) + 1e-8)
) * 100

print("\n===== GRU MODEL METRICS =====")
print(f"MAE   : {mae_gru:.4f}")
print(f"RMSE  : {rmse_gru:.4f}")
print(f"MAPE  : {mape_gru:.2f}%")
print(f"sMAPE : {smape_gru:.2f}%")


In [ ]:
# =========================
# 16. COMPREHENSIVE MODEL COMPARISON
# =========================

comparison_df = pd.DataFrame({
    'Model': ['LSTM Encoder-Decoder', 'GRU Encoder-Decoder', 'Baseline (Lag-24)'],
    'MAE': [f"{mae:.4f}", f"{mae_gru:.4f}", f"{mae_baseline:.4f}"],
    'RMSE': [f"{rmse:.4f}", f"{rmse_gru:.4f}", f"{rmse_baseline:.4f}"],
    'MAPE (%)': [f"{mape:.2f}", f"{mape_gru:.2f}", f"{mape_baseline:.2f}"],
    'sMAPE (%)': [f"{smape:.2f}", f"{smape_gru:.2f}", f"{smape_baseline:.2f}"]
})

print("\n" + "="*100)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

comparison_df.to_csv('models_comprehensive_comparison.csv', index=False)
print(f"\n✓ Saved comprehensive comparison to: models_comprehensive_comparison.csv")

# Identify best model
models_dict = {
    'LSTM': {'mae': mae, 'rmse': rmse, 'mape': mape, 'smape': smape},
    'GRU': {'mae': mae_gru, 'rmse': rmse_gru, 'mape': mape_gru, 'smape': smape_gru},
    'Baseline': {'mae': mae_baseline, 'rmse': rmse_baseline, 'mape': mape_baseline, 'smape': smape_baseline}
}

best_model = min(models_dict, key=lambda x: models_dict[x]['mae'])
print(f"\n🏆 Best Model (by MAE): {best_model}")
print(f"   MAE: {models_dict[best_model]['mae']:.4f}")


In [ ]:
# =========================
# 17. VISUAL COMPARISON: ALL MODELS
# =========================

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = ['LSTM', 'GRU', 'Baseline']
mae_values = [mae, mae_gru, mae_baseline]
rmse_values = [rmse, rmse_gru, rmse_baseline]
mape_values = [mape, mape_gru, mape_baseline]

colors = ['blue', 'green', 'orange']

# MAE comparison
axes[0].bar(models, mae_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('MAE', fontsize=11)
axes[0].set_title('Mean Absolute Error (MAE)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(mae_values):
    axes[0].text(i, v + 0.1, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

# RMSE comparison
axes[1].bar(models, rmse_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('RMSE', fontsize=11)
axes[1].set_title('Root Mean Squared Error (RMSE)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 0.1, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

# MAPE comparison
axes[2].bar(models, mape_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('MAPE (%)', fontsize=11)
axes[2].set_title('Mean Absolute Percentage Error (MAPE)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(mape_values):
    axes[2].text(i, v + 0.2, f'{v:.2f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('models_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved visualization: models_comparison_bars.png")


In [ ]:
# =========================
# 18. FINAL SUMMARY & CONCLUSION
# =========================

print("\n" + "="*100)
print("ELECTRICITY LOAD FORECASTING - PROJECT SUMMARY")
print("="*100)

print(f"\n📊 DATA OVERVIEW:")
print(f"  • Total sequences: {len(df)}")
print(f"  • Training sequences: {len(train_df)}")
print(f"  • Validation sequences: {len(val_df)}")
print(f"  • Test sequences: {len(test_df)}")
print(f"  • Number of features: {n_features}")
print(f"  • Lookback window: {LOOKBACK} hours (7 days)")
print(f"  • Forecast horizon: {HORIZON} hours (1 day)")

print(f"\n🧠 MODEL PERFORMANCE:")
print(f"  LSTM Encoder-Decoder:")
print(f"    • MAE: {mae:.4f} ({improvement_mae:+.2f}% vs baseline)")
print(f"    • RMSE: {rmse:.4f}")
print(f"    • MAPE: {mape:.2f}%")
print(f"")
print(f"  GRU Encoder-Decoder:")
print(f"    • MAE: {mae_gru:.4f}")
print(f"    • RMSE: {rmse_gru:.4f}")
print(f"    • MAPE: {mape_gru:.2f}%")

print(f"\n📁 OUTPUT FILES GENERATED:")
print(f"  ✓ forecast_comparison.png - Visual comparison of predictions vs actuals")
print(f"  ✓ error_per_horizon_comparison.png - Per-horizon error analysis")
print(f"  ✓ training_history.png - Training dynamics over epochs")
print(f"  ✓ models_comparison_bars.png - Bar chart comparison of all models")
print(f"  ✓ predictions_detailed.csv - Detailed predictions with timestamps")
print(f"  ✓ model_results_comparison.csv - Model vs baseline metrics")
print(f"  ✓ models_comprehensive_comparison.csv - All models comparison")

print(f"\n💡 KEY INSIGHTS:")
if improvement_mae > 0:
    print(f"  ✓ Deep learning model significantly outperforms seasonal baseline")
    print(f"  ✓ LSTM captures temporal dependencies better than simple lag-based approach")
else:
    print(f"  ⚠ Model needs further optimization - baseline currently performs better")

error_trend = "increasing" if mape_per_t[-1] > mape_per_t[0] else "decreasing"
print(f"  • Error trend over horizon: {error_trend}")
print(f"  • Best forecast hour: T+{np.argmin(mape_per_t)+1}h")
print(f"  • Worst forecast hour: T+{np.argmax(mape_per_t)+1}h")

print(f"\n✅ RECOMMENDATIONS FOR DEPLOYMENT:")
print(f"  1. Use {'LSTM' if mae <= mae_gru else 'GRU'} model (lower MAE)")
print(f"  2. Retrain monthly with new data to maintain performance")
print(f"  3. Monitor prediction errors in real-world deployment")
print(f"  4. Consider ensemble methods combining LSTM + GRU for robustness")
print(f"  5. Implement alert system for anomalously high prediction errors")

print("\n" + "="*100)
print("END OF ANALYSIS")
print("="*100 + "\n")